# 01f · Separación BCG/ICL con un cutout de la API de TNG (un cúmulo)
**Objetivo:** verificar con datos descargados de la API que el pipeline de `01_` (mismas funciones
de `icl_funcs.py`: banda r, perfil hasta R₂₀₀c, ajuste logarítmico, R_H) da el mismo R_H y f_ICL
para un cúmulo.

- Se descarga el cutout del **halo FoF** (`halos/{FOF_IDX}/cutout.hdf5`) y se seleccionan partículas con
  la máscara por longitudes de subhalo: por defecto **sólo el subhalo central** (definición del paper,
  igual que `01_`); con `INCLUDE_FUZZ = True` se reproduce el chequeo de `01e_`.
- Corte esférico r₃D ≤ R₂₀₀c y sin partículas de viento.
- Las longitudes de subhalo (`SubhaloLenType`) se leen de los archivos locales del Lab.

Requiere haber corrido `00` y `01`.

In [ ]:
# ── Carpeta del repo: el kernel del Lab arranca en /home/tnguser ─────────
import os, sys
from pathlib import Path

REPO_CANDIDATOS = [Path(os.environ.get('ICL_REPO_DIR', Path.cwd())),
                   Path.home() / 'icl_FS' / 'mayes_2026_methodology',
                   Path('/Users/mvs/surface_brightness_profile/mayes_2026_methodology')]
REPO_DIR = next((p.resolve() for p in REPO_CANDIDATOS if (p / 'params_icl.py').is_file()), None)
if REPO_DIR is None:
    raise FileNotFoundError(f"No encuentro params_icl.py en {REPO_CANDIDATOS}; define ICL_REPO_DIR")

os.chdir(REPO_DIR)
# el repo va PRIMERO en sys.path (gana sobre cualquier otro icl_funcs/params_icl)
for p in [REPO_DIR / 'original_shift_code', REPO_DIR]:
    if str(p) in sys.path:
        sys.path.remove(str(p))
    sys.path.insert(0, str(p))
print(f"Carpeta de trabajo: {os.getcwd()}")

In [ ]:
from astropy.cosmology import FlatLambdaCDM

import time
import numpy as np
import h5py
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binned_statistic_2d

import illustris_python as il
import Catalogue
import params_icl as P
import icl_funcs as F

In [ ]:
%matplotlib inline
my_colors = ["#4587f1", "#5fe0ca", "#aa89e9", "#f2cc8f", "#f1b4b4"]
sns.set_theme(style="whitegrid", palette=my_colors)

# estado dinámico: 0 relajado, 1 intermedio, 2 perturbado
COLORS_STATE = {0: my_colors[0], 1: my_colors[3], 2: my_colors[4]}
# componentes: 0 in situ, 1 mergers completados, 2 sobrevivientes
COLORS_COMP  = {0: my_colors[4], 1: my_colors[2], 2: my_colors[1]}


FIG_PDF = './figuras/01f_api_tng_BCG_ICL_R200/pdf'
FIG_PNG = './figuras/01f_api_tng_BCG_ICL_R200/png'
os.makedirs(FIG_PDF, exist_ok=True)
os.makedirs(FIG_PNG, exist_ok=True)


cosmo = FlatLambdaCDM(H0=P.H0, Om0=P.Om0, Ob0=P.Ob0)
print(f"basePath = {P.basePath}")
print(f"SNAP     = {P.SNAP}")
print(f"h        = {P.h}   UL = {P.UL:.4f}   UM = {P.UM:.3e}")

## Configuración — API key y grupo a analizar

In [ ]:
# ── Parámetros principales ──────────────────────────────────────────────
API_KEY   = "4fc61d9b3796c16ec02f0299703478ef"   # ← tu clave de https://www.tng-project.org/users/profile/
GROUP_IDX = 0                   # ← posición en el catálogo ICL local (0-based)
# ────────────────────────────────────────────────────────────────────────

BASE_URL = "https://www.tng-project.org/api/TNG100-1"
SNAP     = P.SNAP   # 99

# Carpeta local donde se guarda el cutout descargado
CUTOUT_DIR = './tng_data'
os.makedirs(CUTOUT_DIR, exist_ok=True)


## Funciones de la API

In [ ]:
def api_get(url, params=None):
    """
    GET a la API de TNG. Devuelve JSON si el Content-Type es JSON,
    o el objeto Response completo si es binario (para cutouts HDF5).
    """
    headers = {"api-key": API_KEY}
    r = requests.get(url, headers=headers, params=params, stream=True)
    r.raise_for_status()
    if 'application/json' in r.headers.get('Content-Type', ''):
        return r.json()
    return r


def download_cutout(url, params, local_path, overwrite=False):
    """
    Descarga un cutout HDF5 de la API y lo guarda en local_path.
    Si ya existe y overwrite=False, lo reutiliza (evita descargas repetidas).

    Retorna la ruta del archivo guardado.
    """
    if os.path.exists(local_path) and not overwrite:
        print(f"  Reutilizando cutout local: {local_path}")
        return local_path

    print(f"  Descargando: {url}")
    r = api_get(url, params=params)

    with open(local_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)

    size_mb = os.path.getsize(local_path) / 1e6
    print(f"  Guardado ({size_mb:.1f} MB): {local_path}")
    return local_path


# Test rápido de conexión
try:
    resp = api_get(f"{BASE_URL}/")
    print(f"Conexión OK — simulación: {resp.get('name', '?')}")
    print(f"  Snapshots disponibles: {resp.get('num_snapshots', '?')}")
except Exception as e:
    print(f"[ERROR] No se pudo conectar a la API: {e}")
    print("  Revisa tu API_KEY y tu conexión a internet.")


## Cargar catálogo local y obtener FOF_IDX, M200c, R200c

In [ ]:
with h5py.File(P.CATALOG_OUT, 'r') as f:
    group_idx_cat = f['group_idx'][:]
    M200c_all     = f['M200c_Msun'][:]
    R200c_all     = f['R200c_kpc'][:]
    GroupPos_all  = f['GroupPos_kpc'][:]
    bcg_sub_all   = f['bcg_sub_idx'][:]
    Mstar200_all  = f['Mstar200_Msun'][:]
    R_H_01_all    = f['R_H_kpc'][:] if 'R_H_kpc' in f else None
    f_icl_01_all  = f['f_icl'][:] if 'f_icl' in f else None
    R_H_fof_all   = f['R_H_kpc_fof'][:] if 'R_H_kpc_fof' in f else None
    f_icl_fof_all = f['f_icl_fof'][:] if 'f_icl_fof' in f else None

FOF_IDX  = int(group_idx_cat[GROUP_IDX])
M200c    = M200c_all[GROUP_IDX]
R200c    = R200c_all[GROUP_IDX]
GroupPos = GroupPos_all[GROUP_IDX]
BCG_sub  = int(bcg_sub_all[GROUP_IDX])
Mstar200 = Mstar200_all[GROUP_IDX]

print(f"GROUP_IDX = {GROUP_IDX}   FOF_IDX = {FOF_IDX}   BCG = {BCG_sub}")
print(f"log M200c = {np.log10(M200c):.2f} Msun   R200c = {R200c:.1f} kpc   log M*200 = {np.log10(Mstar200):.2f}")

## Cargar metadatos del halo desde archivos locales

`GroupFirstSub`, `GroupNsubs` y `SubhaloLenType` no están disponibles de forma fiable en la
respuesta JSON de la API (igual que en `01d_`). Los cargamos directamente desde los archivos
locales con `illustris_python`; la API solo se usa para descargar el cutout de partículas.

In [ ]:
# ── GroupFirstSub, GroupNsubs y BoxSize desde archivos locales ──────────
halo_fields = il.groupcat.loadHalos(P.basePath, P.SNAP,
                                     fields=['GroupFirstSub', 'GroupNsubs'])
first_sub = int(halo_fields['GroupFirstSub'][FOF_IDX])
n_subs    = int(halo_fields['GroupNsubs'][FOF_IDX])

header  = il.groupcat.loadHeader(P.basePath, P.SNAP)
box_kpc = header['BoxSize'] * P.UL   # ckpc/h → kpc físicos

print(f"Halo FoF {FOF_IDX} (cargado localmente):")
print(f"  GroupFirstSub : {first_sub}")
print(f"  GroupNsubs    : {n_subs}")
print(f"  BoxSize       : {box_kpc:.1f} kpc")

# ── SubhaloLenType: una sola carga global + slicing ─────────────────────
# Más rápido que N llamadas a la API — y sin rate-limit
all_lens = il.groupcat.loadSubhalos(P.basePath, P.SNAP,
                                     fields=['SubhaloLenType'])
if all_lens.ndim == 1:
    all_lens = all_lens.reshape(-1, 6)

lens_star = all_lens[first_sub : first_sub + n_subs, 4].astype(int)

print(f"\nSubhaloLenType[4] (estrellas) cargado localmente:")
print(f"  Total en subhalos : {lens_star.sum():,}")
print(f"  Central (sub 0)   : {lens_star[0]:,}")
print(f"  Satélites (1..N-1): {lens_star[1:].sum():,}")


## Descargar cutout del **halo** FoF (no del subhalo)

Usamos `halos/{FOF_IDX}/cutout.hdf5` — equivalente a `loadHalo(...)` en local (central +
satélites + no ligadas). El endpoint `subhalos/{id}/cutout.hdf5` traería **solo** las
partículas ligadas al subhalo central (equivalente a `loadSubhalo`, igual a `01_`, **no**
a `01b_`/`01e_`), así que no lo usamos aquí.

Centramos inmediatamente en la posición del BCG (`GroupPos`) para poder aplicar después
el corte esférico en R₂₀₀c.

In [ ]:
def Distance_1D(X, X_POS, BoxSize):
    """Distancia periódica 1D entre X y X_POS, considerando una caja periódica de tamaño BoxSize."""
    D = X - X_POS
    D = np.where(D >  BoxSize/2, D - BoxSize, D)
    D = np.where(D < -BoxSize/2, D + BoxSize, D)
    return D


cutout_fields = 'Coordinates,Masses,GFM_StellarPhotometrics,GFM_StellarFormationTime'

# URL del cutout del HALO FoF completo (NO subhalos/)
cutout_url   = f"{BASE_URL}/snapshots/{SNAP}/halos/{FOF_IDX}/cutout.hdf5"
cutout_local = f"{CUTOUT_DIR}/cutout_halo_{FOF_IDX}_snap{SNAP}.hdf5"

cutout_path = download_cutout(
    cutout_url,
    params={'stars': cutout_fields},
    local_path=cutout_local,
    overwrite=False   # ← True para forzar re-descarga
)

# Leer el archivo HDF5 descargado
with h5py.File(cutout_path, 'r') as f:
    keys         = list(f['PartType4'].keys())
    pos_raw_ckpc = f['PartType4']['Coordinates'][:]              # ckpc/h
    mass_raw     = f['PartType4']['Masses'][:]                   # 1e10 M_sun/h
    phot_raw     = f['PartType4']['GFM_StellarPhotometrics'][:]  # mag Vega/AB
    sft_raw      = (f['PartType4']['GFM_StellarFormationTime'][:]
                    if 'GFM_StellarFormationTime' in f['PartType4'] else None)

# Convertir unidades y centrar en el BCG (periodicidad incluida)
pos_kpc  = Distance_1D(pos_raw_ckpc, 0, P.L_BOX) * P.UL   # ckpc/h → kpc físicos
pos_c    = Catalogue.Distance_3D(pos_kpc, GroupPos, box_kpc)
mass_raw = mass_raw * P.UM     # M_sun

if phot_raw.ndim == 1:
    phot_raw = phot_raw.reshape(-1, 8)

n_fof_total = len(mass_raw)
print(f"\nPartículas estelares en el cutout FoF: {n_fof_total:,}")
print(f"Campos disponibles en PartType4: {keys}")

## Máscara por longitudes de subhalo (`F.mask_from_lens`, la misma de `01e_`)
Las partículas del cutout FoF vienen ordenadas `[subhalo 0] [subhalo 1] ... [subhalo N-1] [no ligadas]`.

In [ ]:
INCLUDE_SATELLITES = False   # satélites fuera (paper)
INCLUDE_FUZZ       = False   # False = sólo subhalo central (= 01_);  True = central + fuzz (= 01e_)

mask_sub = F.mask_from_lens(lens_star, n_fof_total, INCLUDE_SATELLITES, INCLUDE_FUZZ)

print(f"Total FoF       : {n_fof_total:>9,}")
print(f"Subhalo central : {int(lens_star[0]):>9,}")
print(f"Satélites       : {int(lens_star[1:].sum()):>9,}  ({'incluidos' if INCLUDE_SATELLITES else 'excluidos'})")
print(f"No ligadas      : {n_fof_total - int(lens_star.sum()):>9,}  ({'incluidas' if INCLUDE_FUZZ else 'excluidas'})")
print(f"Seleccionadas   : {mask_sub.sum():>9,}")

## Corte esférico r₃D ≤ R₂₀₀c y partículas de viento

In [ ]:
r_3d      = np.linalg.norm(pos_c, axis=1)
mask_r200 = r_3d <= P.R_OUTER_FRAC * R200c
if sft_raw is not None:
    mask_star = sft_raw > 0
else:
    mask_star = np.ones(n_fof_total, dtype=bool)
    print("[AVISO] el cutout no trae GFM_StellarFormationTime: vuelve a descargarlo con overwrite=True")

mask_final = mask_sub & mask_r200 & mask_star
stars_api  = {'pos': pos_c[mask_final], 'mass': mass_raw[mask_final], 'phot': phot_raw[mask_final]}

print(f"Fuera de R200    : {(mask_sub & ~mask_r200).sum():>9,}")
print(f"Viento           : {(mask_sub & mask_r200 & ~mask_star).sum():>9,}")
print(f"Usadas (01f_)    : {mask_final.sum():>9,}")

## Funciones
Rotación, perfil μ_r(R), ajuste logarítmico y R_H vienen de `icl_funcs.py` (las mismas de `01_`, `01e_` y `03_`).

## Análisis: perfil de Holmberg y separación BCG/ICL (R₂₀₀c)

In [ ]:
res_api = F.bcg_icl_from_stars(stars_api, Mstar200, R200c, P)
pos_rot = res_api['pos_rot']

print(f"R_H (ajuste)    : {res_api['R_H_fit']:.1f} kpc    R_H (interp): {res_api['R_H_interp']:.1f} kpc")
print(f"M_BCG           : {res_api['M_bcg']:.3e} Msun")
print(f"M_ICL           : {res_api['M_icl']:.3e} Msun")
print(f"f_ICL (M*200)   : {res_api['f_icl']:.4f}")
print(f"f_ICL (BCG+ICL) : {res_api['f_icl_bcgicl']:.4f}")

ref = (R_H_fof_all, f_icl_fof_all, '01e_') if INCLUDE_FUZZ else (R_H_01_all, f_icl_01_all, '01_')
if ref[0] is not None:
    print(f"\nComparación con {ref[2]} (mismo cúmulo, archivos locales):")
    print(f"  R_H   : {ref[0][GROUP_IDX]:.1f} kpc  vs API {res_api['R_H']:.1f} kpc")
    print(f"  f_ICL : {ref[1][GROUP_IDX]:.4f}      vs API {res_api['f_icl']:.4f}")
    print("  (deben coincidir si el cutout de la API = loadHalo/loadSubhalo local)")
else:
    print(f"\nNo hay resultados de {ref[2]} en el catálogo: córrelo primero")

### Figura 1 – Perfil de brillo superficial 1D

In [ ]:
rr = np.linspace(1, R200c, 500)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(res_api['r_mid'], res_api['mu'], 'o', ms=4, color=my_colors[0], label='Perfil 1D (API)')
ax.plot(rr, res_api['a'] + res_api['b'] * np.log10(rr), 'k-', lw=1.5, label='Ajuste logarítmico')
ax.axhline(P.MU_HOLMBERG, color='gray', ls='--', lw=1)
ax.axvline(res_api['R_H'], color='r', lw=1.5, label=f"R_H = {res_api['R_H']:.0f} kpc")
ax.set_ylim(33, 15)
ax.set_xlabel('Radio [kpc]')
ax.set_ylabel(r'$\mu_r$ [mag arcsec$^{-2}$]')
ax.set_title(f'FoF {FOF_IDX}  (cutout API, snap {SNAP})')
ax.legend()

plt.savefig(f'{FIG_PDF}/fig01f_perfil_sb_fof{FOF_IDX}_snap{SNAP}.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig01f_perfil_sb_fof{FOF_IDX}_snap{SNAP}.png', dpi=150, bbox_inches='tight')
plt.show()

### Figura 2 – Mapa 2D de brillo superficial

In [ ]:
r_plot  = R200c
n_pix   = 512
edges   = np.linspace(-r_plot, r_plot, n_pix + 1)
pix_pc2 = ((2 * r_plot / n_pix) * 1e3)**2

L_map, _, _, _ = binned_statistic_2d(pos_rot[:, 0], pos_rot[:, 1], res_api['lum'],
                                     statistic='sum', bins=[edges, edges])
mu_map = F.mu_from_sigma(L_map / pix_pc2, P.SB_CONST)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mu_map.T, origin='lower', cmap='ocean', vmin=16, vmax=32,
               extent=[-r_plot, r_plot, -r_plot, r_plot])
plt.colorbar(im, ax=ax, label=r'$\mu_r$ [mag arcsec$^{-2}$]')
ax.add_patch(plt.Circle((0, 0), res_api['R_H'], fill=False, color='k', lw=1.5, label='R_H'))
ax.add_patch(plt.Circle((0, 0), R200c, fill=False, color='gray', lw=1, ls='--', label='R$_{200}$'))
ax.set_xlabel('x [kpc]')
ax.set_ylabel('y [kpc]')
ax.grid(False)
ax.legend(loc='upper right')

plt.savefig(f'{FIG_PDF}/fig01f_mapa_sb_fof{FOF_IDX}_snap{SNAP}.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig01f_mapa_sb_fof{FOF_IDX}_snap{SNAP}.png', dpi=150, bbox_inches='tight')
plt.show()